In [3]:
# CELL 1: Environment setup and load Stage 3 triples
import json
import pandas as pd
import networkx as nx
from pathlib import Path
from collections import Counter

RESULTS_DIR = Path("../data/results")
TRIPLES_FILE    = RESULTS_DIR / "community_triples.json"
GRAPH_FILE      = RESULTS_DIR / "knowledge_graph.graphml"
NODE_FILE       = RESULTS_DIR / "knowledge_graph_nodes.csv"
EDGE_FILE       = RESULTS_DIR / "knowledge_graph_edges.csv"
METRICS_FILE    = RESULTS_DIR / "knowledge_graph_metrics.json"

with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

print(f"Loaded triples from {len(community_triples)} communities")
print("Sample:", list(community_triples.items())[0])

Loaded triples from 19 communities
Sample: ('0', [{'subject': 'ftp_client', 'relation': 'targets', 'target': 'ftp_port_21_service'}, {'subject': 'ftp_client', 'relation': 'scans', 'target': 'http_web_server'}, {'subject': 'http_flood_source', 'relation': 'floods', 'target': 'http_web_server'}, {'subject': 'http_flood_source', 'relation': 'targets', 'target': 'http_port_80_service'}])


In [4]:
# CELL 2: Node type inference
# With open-entity triples, node names are descriptive strings extracted by the LLM.
# We infer node type heuristically from the name content.
# This is now meaningful: nodes represent real discovered entities, not enum values.

def infer_node_type(node: str) -> str:
    n = node.lower()
    if any(x in n for x in ["scanner", "attacker", "client", "source", "actor", "process", "bot"]):
        return "threat_actor"
    if any(x in n for x in ["port", "service", "server", "endpoint", "http", "ssh", "ftp", "dns"]):
        return "service_target"
    if any(x in n for x in ["attack", "exploit", "brute", "flood", "spray", "scan", "injection"]):
        return "attack_technique"
    if any(x in n for x in ["traffic", "connection", "flow", "packet", "volume", "duration"]):
        return "network_behavior"
    return "entity"

In [5]:
# CELL 3: Construct directed knowledge graph
#
# With open triples, edges now carry real semantic content:
# "ssh_brute_force_client --[targets]--> ssh_port_22_service" is meaningful.
# This is what the professor means by "edges carry meaning."
#
# Edge weight = how many communities produced the same triple.
# A high-weight edge means that relationship appears repeatedly across communities
# — that is a signal, not noise.

G = nx.DiGraph()
edge_weights = {}  # (src, rel, tgt) -> count
total_triples = 0
invalid_triples = 0

# Also track which communities each edge came from (useful for RAG provenance)
edge_communities = {}

for cid, triples in community_triples.items():
    for t in triples:
        s = str(t.get("subject", "")).strip()
        r = str(t.get("relation", "")).strip()
        o = str(t.get("target", "")).strip()

        if not s or not r or not o:
            invalid_triples += 1
            continue

        total_triples += 1
        key = (s, r, o)
        edge_weights[key] = edge_weights.get(key, 0) + 1
        edge_communities.setdefault(key, []).append(cid)

for (src, rel, tgt), weight in edge_weights.items():
    G.add_node(src, type=infer_node_type(src))
    G.add_node(tgt, type=infer_node_type(tgt))
    G.add_edge(
        src, tgt,
        relation=rel,
        weight=weight,
        communities=",".join(edge_communities[(src, rel, tgt)])
    )

print(f"Graph constructed: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Processed {total_triples} triples | Skipped {invalid_triples} invalid")
print(f"\nNote: With open extraction you should see many more nodes than the")
print(f"previous version (which had only 12 nodes from a closed enum).") 

Graph constructed: 38 nodes, 43 edges
Processed 76 triples | Skipped 0 invalid

Note: With open extraction you should see many more nodes than the
previous version (which had only 12 nodes from a closed enum).


In [6]:
# CELL 4: Inspect the most connected nodes — these are your high-value findings
# In a well-extracted knowledge graph, high-degree nodes are entities that
# appear in many relationships across communities. These are analytically interesting.

print("=== TOP 10 NODES BY DEGREE ===")
degree_sorted = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
for node, deg in degree_sorted:
    ntype = G.nodes[node].get('type', 'unknown')
    print(f"  [{ntype:20s}] {node} (degree={deg})")

print("\n=== TOP 10 EDGES BY WEIGHT (most recurring relationships) ===")
edge_data = [(u, v, d['relation'], d['weight']) for u, v, d in G.edges(data=True)]
edge_data_sorted = sorted(edge_data, key=lambda x: x[3], reverse=True)[:10]
for u, v, rel, w in edge_data_sorted:
    print(f"  (weight={w}) {u} --[{rel}]--> {v}")

=== TOP 10 NODES BY DEGREE ===
  [threat_actor        ] ftp_client (degree=10)
  [service_target      ] http_web_server (degree=7)
  [service_target      ] dns_resolver (degree=7)
  [threat_actor        ] http_flood_source (degree=5)
  [service_target      ] ssh_port_22_service (degree=5)
  [network_behavior    ] high_volume_short_connections (degree=5)
  [threat_actor        ] ssh_brute_force_client (degree=3)
  [threat_actor        ] dns_client (degree=3)
  [service_target      ] ssh_authentication_endpoint (degree=3)
  [threat_actor        ] https_client (degree=3)

=== TOP 10 EDGES BY WEIGHT (most recurring relationships) ===
  (weight=3) dns_resolver --[scans]--> http_web_server
  (weight=2) ftp_client --[scans]--> ftp_port_21_service
  (weight=2) ftp_client --[floods]--> network_bandwidth
  (weight=2) ssh_brute_force_client --[scans]--> ssh_port_22_service
  (weight=2) credential_guessing_process --[executes]--> password_spray_attack
  (weight=2) dns_client --[scans]--> dns_resol

In [7]:
# CELL 5: Export graph structure
nx.write_graphml(G, GRAPH_FILE)

nodes_out = [
    {"node": n, "type": G.nodes[n].get("type", "unknown"), "degree": d}
    for n, d in G.degree()
]
pd.DataFrame(nodes_out).sort_values("degree", ascending=False).to_csv(NODE_FILE, index=False)

edges_out = [
    {"source": u, "target": v,
     "relation": data["relation"],
     "weight": data["weight"],
     "communities": data.get("communities", "")}
    for u, v, data in G.edges(data=True)
]
pd.DataFrame(edges_out).sort_values("weight", ascending=False).to_csv(EDGE_FILE, index=False)

print("Exported graph to GraphML, nodes.csv, edges.csv")

Exported graph to GraphML, nodes.csv, edges.csv


In [8]:
# CELL 6: Compute and save graph metrics
relation_dist = Counter([data["relation"] for _, _, data in G.edges(data=True)])
type_dist     = Counter([G.nodes[n].get("type", "unknown") for n in G.nodes()])

metrics = {
    "construction_method": "LLM open-extraction (subject, relation, target) triples -> NetworkX DiGraph",
    "extraction_model": "qwen2.5-8b-instruct-q4_k_m",
    "total_triples_processed": total_triples,
    "invalid_triples_skipped": invalid_triples,
    "unique_nodes": G.number_of_nodes(),
    "unique_edges": G.number_of_edges(),
    "average_degree": round(sum(dict(G.degree()).values()) / max(G.number_of_nodes(), 1), 3),
    "relation_distribution": dict(relation_dist),
    "node_type_distribution": dict(type_dist),
    # These two metrics are the key differentiator from the old enum-based approach:
    # unique_nodes should now be >> 12
    # relation_distribution should reflect varied attack behavior patterns
}

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("=== KNOWLEDGE GRAPH METRICS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")
print("\nStage 4 complete.")

=== KNOWLEDGE GRAPH METRICS ===
  construction_method: LLM open-extraction (subject, relation, target) triples -> NetworkX DiGraph
  extraction_model: qwen2.5-8b-instruct-q4_k_m
  total_triples_processed: 76
  invalid_triples_skipped: 0
  unique_nodes: 38
  unique_edges: 43
  average_degree: 2.263
  relation_distribution: {'scans': 12, 'targets': 11, 'executes': 8, 'floods': 5, 'generates': 5, 'authenticates_to': 2}
  node_type_distribution: {'threat_actor': 9, 'service_target': 23, 'attack_technique': 2, 'entity': 1, 'network_behavior': 3}

Stage 4 complete.
